In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from math import sqrt

class KNNScratch:
    """
    K-Nearest Neighbors implémenté from scratch
    Bibliothèques utilisées: numpy, pandas, matplotlib (pas sklearn)
    """
    
    def __init__(self, k=3, distance_metric='euclidean'):
        """
        Parameters:
        -----------
        k : int - Nombre de voisins
        distance_metric : str - 'euclidean', 'manhattan', 'minkowski'
        """
        self.k = k
        self.distance_metric = distance_metric
        self.X_train = None
        self.y_train = None
        
    def _calculate_distance(self, x1, x2, p=3):
        """
        Calcule la distance entre deux points
        """
        if self.distance_metric == 'euclidean':
            return np.sqrt(np.sum((x1 - x2) ** 2))
        
        elif self.distance_metric == 'manhattan':
            return np.sum(np.abs(x1 - x2))
        
        elif self.distance_metric == 'minkowski':
            return np.sum(np.abs(x1 - x2) ** p) ** (1/p)
        
        else:
            raise ValueError(f"Métrique {self.distance_metric} non supportée")
    
    def fit(self, X, y):
        """
        Stocke les données d'entraînement
        """
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        
    def _predict_one(self, x):
        """
        Prédit la classe pour un seul échantillon
        """
        # Calculer les distances
        distances = []
        for i in range(len(self.X_train)):
            dist = self._calculate_distance(x, self.X_train[i])
            distances.append((dist, self.y_train[i]))
        
        # Trier par distance
        distances.sort(key=lambda t: t[0])
        
        # Prendre les k plus proches
        k_neighbors = distances[:self.k]
        
        # Vote majoritaire
        votes = [label for _, label in k_neighbors]
        most_common = Counter(votes).most_common(1)
        
        return most_common[0][0]
    
    def predict(self, X):
        """
        Prédit les classes pour plusieurs échantillons
        """
        X = np.array(X)
        predictions = [self._predict_one(x) for x in X]
        return np.array(predictions)
    
    def score(self, X, y):
        """
        Calcule l'accuracy
        """
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    
    def confusion_matrix(self, X, y):
        """
        Calcule la matrice de confusion
        """
        y_pred = self.predict(X)
        classes = np.unique(np.concatenate([y, y_pred]))
        n_classes = len(classes)
        
        # Dictionnaire classe -> index
        class_to_idx = {cls: i for i, cls in enumerate(classes)}
        
        # Initialiser la matrice
        cm = np.zeros((n_classes, n_classes), dtype=int)
        
        # Remplir
        for true, pred in zip(y, y_pred):
            cm[class_to_idx[true]][class_to_idx[pred]] += 1
        
        return cm, classes


def creer_dataset_synthetique(n_samples=300, n_features=2, n_classes=3, random_state=42):
    """
    Crée un dataset synthétique avec numpy
    """
    np.random.seed(random_state)
    
    X_list = []
    y_list = []
    
    # Centres des classes
    centers = {
        0: np.array([2, 2]),
        1: np.array([-2, -2]),
        2: np.array([2, -2])
    }
    
    for classe in range(n_classes):
        # Générer des points autour du centre
        X_class = np.random.randn(n_samples // n_classes, n_features) + centers[classe]
        y_class = np.full(n_samples // n_classes, classe)
        
        X_list.append(X_class)
        y_list.append(y_class)
    
    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    
    return X, y


def train_test_split_custom(X, y, test_size=0.2, random_state=42):
    """
    Division manuelle train/test
    """
    np.random.seed(random_state)
    n_samples = len(X)
    indices = np.random.permutation(n_samples)
    
    test_size_int = int(n_samples * test_size)
    test_indices = indices[:test_size_int]
    train_indices = indices[test_size_int:]
    
    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]
    
    return X_train, X_test, y_train, y_test


def visualiser_frontiere_decision(knn, X, y, title="Frontière de décision KNN"):
    """
    Visualise la frontière de décision (pour 2D)
    """
    if X.shape[1] != 2:
        print("La visualisation nécessite des données 2D")
        return
    
    # Créer la grille
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    # Prédire sur la grille
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Afficher
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='Set3')
    
    # Afficher les points
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='Set1', edgecolors='black', s=50)
    plt.colorbar(scatter)
    plt.xlabel('Caractéristique 1')
    plt.ylabel('Caractéristique 2')
    plt.title(title)
    plt.show()


def trouver_meilleur_k(X_train, y_train, X_val, y_val, k_max=20):
    """
    Trouve la meilleure valeur de k
    """
    k_values = range(1, k_max + 1)
    accuracies = []
    
    for k in k_values:
        knn = KNNScratch(k=k)
        knn.fit(X_train, y_train)
        acc = knn.score(X_val, y_val)
        accuracies.append(acc)
    
    best_k = k_values[np.argmax(accuracies)]
    
    return best_k, accuracies


# ============================================
# Programme principal
# ============================================

if __name__ == "__main__":
    
    print("="*60)
    print("KNN FROM SCRATCH - NumPy, Pandas, Matplotlib")
    print("="*60)
    
    # 1. Création du dataset
    print("\n1. Création du dataset synthétique...")
    X, y = creer_dataset_synthetique(n_samples=300, n_features=2, n_classes=3)
    
    # Convertir en DataFrame pandas pour l'affichage
    df = pd.DataFrame(X, columns=['x1', 'x2'])
    df['classe'] = y
    print(df.head())
    print(f"\nShape: {df.shape}")
    print(f"Distribution des classes:\n{df['classe'].value_counts()}")
    
    # 2. Visualisation initiale des données
    print("\n2. Visualisation des données...")
    plt.figure(figsize=(8, 6))
    colors = {0: 'red', 1: 'blue', 2: 'green'}
    for classe in np.unique(y):
        mask = y == classe
        plt.scatter(X[mask, 0], X[mask, 1], c=colors[classe], label=f'Classe {classe}', alpha=0.7)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('Distribution des données')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # 3. Division train/test/validation
    print("\n3. Division des données...")
    X_train, X_temp, y_train, y_temp = train_test_split_custom(X, y, test_size=0.3)
    X_val, X_test, y_val, y_test = train_test_split_custom(X_temp, y_temp, test_size=0.5)
    
    print(f"Train: {len(X_train)} samples")
    print(f"Validation: {len(X_val)} samples")
    print(f"Test: {len(X_test)} samples")
    
    # 4. Trouver le meilleur k
    print("\n4. Recherche du meilleur k...")
    best_k, accuracies = trouver_meilleur_k(X_train, y_train, X_val, y_val, k_max=15)
    
    # Visualiser les accuracies
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, 16), accuracies, marker='o', linewidth=2, markersize=8)
    plt.xlabel('k', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('Accuracy en fonction de k (validation)', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.axvline(x=best_k, color='red', linestyle='--', label=f'Meilleur k={best_k}')
    plt.legend()
    plt.show()
    
    print(f"Meilleur k trouvé: {best_k}")
    
    # 5. Entraîner avec le meilleur k
    print("\n5. Entraînement du modèle final...")
    knn_final = KNNScratch(k=best_k, distance_metric='euclidean')
    knn_final.fit(X_train, y_train)
    
    # 6. Évaluation sur test
    print("\n6. Évaluation sur l'ensemble de test...")
    y_pred = knn_final.predict(X_test)
    accuracy = knn_final.score(X_test, y_test)
    print(f"Accuracy sur test: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    # 7. Matrice de confusion
    print("\n7. Matrice de confusion...")
    cm, classes = knn_final.confusion_matrix(X_test, y_test)
    
    # Afficher matrice de confusion avec pandas
    cm_df = pd.DataFrame(cm, 
                         index=[f'Classe {c}' for c in classes], 
                         columns=[f'Classe {c}' for c in classes])
    print("\nMatrice de confusion:")
    print(cm_df)
    
    # Visualiser matrice de confusion
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar()
    plt.title('Matrice de Confusion')
    plt.xlabel('Prédit')
    plt.ylabel('Réel')
    
    # Ajouter les valeurs dans les cases
    for i in range(len(classes)):
        for j in range(len(classes)):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')
    
    plt.xticks(range(len(classes)), [f'Classe {c}' for c in classes])
    plt.yticks(range(len(classes)), [f'Classe {c}' for c in classes])
    plt.show()
    
    # 8. Visualiser la frontière de décision
    print("\n8. Visualisation de la frontière de décision...")
    visualiser_frontiere_decision(knn_final, X, y, "Frontière de décision KNN")
    
    # 9. Comparaison des métriques
    print("\n9. Comparaison des métriques de distance...")
    metrics = ['euclidean', 'manhattan', 'minkowski']
    results = []
    
    for metric in metrics:
        knn_metric = KNNScratch(k=best_k, distance_metric=metric)
        knn_metric.fit(X_train, y_train)
        acc = knn_metric.score(X_test, y_test)
        results.append(acc)
        print(f"  {metric:10s}: accuracy = {acc:.4f}")
    
    # Graphique comparatif
    plt.figure(figsize=(8, 5))
    bars = plt.bar(metrics, results, color=['blue', 'green', 'orange'])
    plt.ylabel('Accuracy')
    plt.title('Comparaison des métriques de distance')
    plt.ylim([0, 1])
    
    # Ajouter les valeurs sur les barres
    for bar, acc in zip(bars, results):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{acc:.3f}', ha='center', va='bottom')
    
    plt.show()
    
    # 10. Analyse détaillée des erreurs
    print("\n10. Analyse des erreurs...")
    errors = np.where(y_pred != y_test)[0]
    print(f"Nombre d'erreurs: {len(errors)} sur {len(y_test)} ({len(errors)/len(y_test)*100:.2f}%)")
    
    if len(errors) > 0:
        print("\nExemples d'erreurs:")
        for i in errors[:5]:
            print(f"  Échantillon {i}: Vrai={y_test[i]}, Prédit={y_pred[i]}")
    
    # 11. Prédictions avec probabilités (approximation)
    print("\n11. Test sur nouveaux points...")
    nouveaux_points = np.array([
        [0, 0],
        [3, 3],
        [-3, -3],
        [2, -2]
    ])
    
    predictions = knn_final.predict(nouveaux_points)
    
    # Afficher résultats
    for i, point in enumerate(nouveaux_points):
        print(f"Point {point} -> Classe prédite: {predictions[i]}")
    
    # Visualiser les nouveaux points
    plt.figure(figsize=(10, 8))
    # Données originales
    for classe in np.unique(y):
        mask = y == classe
        plt.scatter(X[mask, 0], X[mask, 1], c=colors[classe], label=f'Classe {classe}', alpha=0.5, s=30)
    
    # Nouveaux points
    plt.scatter(nouveaux_points[:, 0], nouveaux_points[:, 1], 
               c='black', marker='*', s=200, label='Nouveaux points', edgecolors='white', linewidth=2)
    
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('Classification de nouveaux points')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("\n" + "="*60)
    print("RÉSUMÉ FINAL")
    print("="*60)
    print(f"Meilleur k: {best_k}")
    print(f"Accuracy finale: {accuracy:.4f}")
    print(f"Nombre d'erreurs: {len(errors)}")
    print("="*60)